> **NOTICE (Apache License 2.0, section 4b)** — This notebook is a **modified**
> derivative of `vehicles-detection-and-counting.ipynb` (Apache License 2.0).
> Original: [original author, if known] — [original source URL].
> Changes made: detector upgraded YOLOv8 → YOLO26; logic extracted into
> `vd_lib26.py`; BoT-SORT tracking / shadow filtering / lane-specific geometry
> removed; format-neutral function names; ground-truth evaluation section
> added; ffmpeg & memory bugs fixed. Full list in `NOTICE.md`; license copy in
> `LICENSE`.

**Derived work (Apache License 2.0).** This notebook is a heavily modified version of `vehicles-detection-and-counting.ipynb` (Apache License 2.0). The full list of changes is in `README.md` → "Changes from the original notebook". License: Apache 2.0 — see `LICENSE`.

- **Inputs:** `data_input/sample_video_copy.mp4` (the video to analyze)


In [ ]:
import os
from pathlib import Path

# Auto-locate the repo root: the nearest ancestor of the working directory
# that contains vd_lib26.py. Works without editing anything - just open the
# notebook from the repo folder (the standard Jupyter flow) and ROOT is right.
def _repo_root(marker="vd_lib26.py"):
    p = Path.cwd().resolve()
    for d in (p, *p.parents):
        if (d / marker).exists():
            return d
    return p

ROOT = _repo_root()
DATA_OUT   = ROOT / "data_out"
TRAIN_OUT  = ROOT / "train_out26"
MODEL_PATH = ROOT / "best26.pt"          # fine-tuned model copy in this repo folder
SRC_VIDEO  = ROOT / "data_input" / "sample_video_copy.mp4"   # the video to analyze
os.makedirs(DATA_OUT, exist_ok=True)
os.makedirs(TRAIN_OUT, exist_ok=True)

In [ ]:
# %pip install -q ultralytics kagglehub pandas seaborn scipy opencv-python matplotlib
# Load every function from vd_lib26.py (sits next to this notebook).
get_ipython().run_line_magic("run", str(ROOT / "vd_lib26.py"))

## 1. Pre-trained model (COCO)

In [ ]:
model = load_model("yolo26s.pt")   # pre-trained YOLO26s; auto-downloaded on first use
model

In [ ]:
# Fine-tuning dataset (top-view vehicles) - auto-downloaded from Kaggle on first run.
# Requires a free Kaggle account: locally kagglehub prompts once; on Colab set the
# KAGGLE_USERNAME / KAGGLE_KEY secrets (or upload kaggle.json) first.
import kagglehub
dataset_root = kagglehub.dataset_download("farzadnekouei/top-view-vehicle-detection-image-dataset")
dataset_path = os.path.join(dataset_root, "Vehicle_Detection_Image_Dataset")
print("Dataset ready at:", dataset_path)

In [ ]:
import matplotlib.pyplot as plt
sample_image_path = os.path.join(dataset_path, "sample_image.jpg")

plt.figure(figsize=(20, 15))
plt.imshow(infer_image(model, sample_image_path, conf=0.5))
plt.axis("off"); plt.title("Pre-trained YOLO26 (COCO) - sample image"); plt.show()

## 2. Fine-tuning dataset (top-view vehicles)

In [ ]:
import yaml
with open(os.path.join(dataset_path, "data.yaml")) as f:
    print(yaml.dump(yaml.safe_load(f), default_flow_style=False))

from PIL import Image
def count_sizes(sub):
    p = os.path.join(dataset_path, sub, "images")
    n, sizes = 0, set()
    for fn in os.listdir(p):
        if fn.endswith(".jpg"):
            n += 1
            with Image.open(os.path.join(p, fn)) as im:
                sizes.add(im.size)
    return n, sizes
for sub in ("train", "valid"):
    n, sizes = count_sizes(sub)
    print(f"{sub}: {n} images, unique sizes {sizes}")

## 3. Fine-tuning (YOLO26s)

Trains on the best available device automatically - CUDA on Colab, MPS on Apple Silicon, CPU otherwise (~30-60 min for 30 epochs on an M1; faster on Colab's free GPU).

Cell 10 copies the resulting weights to `best26.pt` (the file `MODEL_PATH` points to). **On Colab, download `best26.pt` (and `data_out/` if you need it) before the session ends** - the VM is wiped afterwards.

In [ ]:
# %pip install -U ultralytics   # YOLO26 training needs ultralytics >= 8.4.0

# Pick the best available device automatically: CUDA (Colab GPU),
# MPS (Apple Silicon), or CPU - works on any machine without edits.
import torch
device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print("Training on:", device)

# Fine-tune YOLO26s on the top-view vehicle dataset
results = model.train(
    data=os.path.join(dataset_path, "data.yaml"),
    epochs=30,
    imgsz=640,
    batch=8,               # modest; fine on both M1 and Colab T4
    device=device,
    workers=2,
    patience=20,
    project=str(TRAIN_OUT),
    name="train",
    exist_ok=True,
)

# Copy the fine-tuned weights to where the rest of the notebook expects them:
import shutil
shutil.copy(os.path.join(TRAIN_OUT, "train", "weights", "best.pt"), MODEL_PATH)
print("best26.pt ready at", MODEL_PATH)

## 4. Fine-tuned model - inference

In [ ]:
best_model = load_model(MODEL_PATH)   # uses the copy in this repo folder, not the training output

In [ ]:
valid_dir = os.path.join(dataset_path, "valid", "images")
files = sorted(f for f in os.listdir(valid_dir) if f.endswith(".jpg"))
files = [files[i] for i in range(0, len(files), len(files) // 9)][:9]

fig, axes = plt.subplots(3, 3, figsize=(20, 21))
for ax, fn in zip(axes.ravel(), files):
    ax.imshow(infer_image(best_model, os.path.join(valid_dir, fn), conf=0.5, line_width=1))
    ax.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
plt.figure(figsize=(20, 15))
plt.imshow(infer_image(best_model, sample_image_path, conf=0.7))
plt.axis("off"); plt.title("Fine-tuned YOLO26 - unseen test image"); plt.show()

## 5. Sample video - detection & analysis

**Note:** the detection slice in `vd_lib26` (`X1`/`X2`, the blacked-out regions) was tuned for the original 1280x720 sample camera. Videos with a different resolution may need those constants adjusted in `vd_lib26.py` (constants section) and a re-run of cell 2.

In [ ]:
src_video = str(SRC_VIDEO)
print("Analyzing:", src_video)
infer_video_save(best_model, src_video, project=DATA_OUT, name="predict1", exist_ok=True)

### 5.1 Real-time vehicle counting (per frame)

In [ ]:
n_frames = count_vehicles_in_video(
    best_model, src_video,
    output_avi=os.path.join(DATA_OUT, "vehicle_count.avi"),
    conf=0.4)
print("processed", n_frames, "frames")

## 6. VisDrone test video - accuracy vs frame

Runs `count_vehicles_in_video` on the VisDrone clip in `data_input/test_video_visdrone/` (780 frames + ground-truth annotations) and plots the per-frame count accuracy against the annotations.

- Ground truth = VisDrone boxes for classes car / van / truck / bus
- The detection slice is disabled (`x1=0, x2=height`) because VisDrone frames are not 1280x720
- Run this with the fine-tuned model (cell 12) for a meaningful number

In [ ]:
# --- VisDrone test clip: full evaluation (ground truth, results video, accuracy) ---
vd_dir = ROOT / "data_input" / "test_video_visdrone"
evaluate_sequence(
    best_model,
    vd_dir / "uav0000077_00720_v",
    vd_dir / "uav0000077_00720_v.txt",
    DATA_OUT,
)